## polls_questionset 질문 세트 테이블 전처리 확인

In [1]:
import json
from numbers import Number
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"
TABLE_NAME = "polls_questionset"

client = bigquery.Client(project=PROJECT_ID)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [2]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.{TABLE_NAME}`
"""

df = client.query(sql).to_dataframe()
df.head()

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,question_piece_id_list,opening_time,status,created_at,user_id
0,7993167,"[79931958, 79931960, 79931962, 79931966, 79931...",2023-05-17 00:22:16+00:00,C,2023-05-16 23:42:16+00:00,841333
1,8298931,"[82989582, 82989587, 82989592, 82989596, 82989...",2023-05-17 09:54:57+00:00,C,2023-05-17 09:14:57+00:00,845326
2,20617202,"[206172781, 206172782, 206172783, 206172784, 2...",2023-07-29 16:13:18+00:00,C,2023-07-29 15:33:18+00:00,847375
3,20017608,"[200176841, 200176842, 200176843, 200176844, 2...",2023-06-16 15:13:10+00:00,C,2023-06-16 14:33:10+00:00,849438
4,20673673,"[206737491, 206737492, 206737493, 206737494, 2...",2023-08-10 09:20:44+00:00,C,2023-08-10 08:40:44+00:00,849451


## 결측치 및 데이터 정보 확인

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 158384 entries, 0 to 158383
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   id                      158384 non-null  Int64              
 1   question_piece_id_list  158384 non-null  str                
 2   opening_time            158384 non-null  datetime64[us, UTC]
 3   status                  158384 non-null  str                
 4   created_at              158384 non-null  datetime64[us, UTC]
 5   user_id                 158384 non-null  Int64              
dtypes: Int64(2), datetime64[us, UTC](2), str(2)
memory usage: 23.0 MB


In [4]:
df.isna().sum()

id                        0
question_piece_id_list    0
opening_time              0
status                    0
created_at                0
user_id                   0
dtype: int64

## 중복값 확인

In [5]:
duplicate_rows = df.assign(question_piece_id_list=df["question_piece_id_list"].astype("string")).duplicated().sum()
print("전체 행 중복:", duplicate_rows)
print("id 중복:", df["id"].duplicated().sum())

전체 행 중복: 0
id 중복: 0


## 날짜 범위 및 미래 날짜 확인

In [6]:
date_cols = ["created_at", "opening_time"]
display(df[date_cols].agg(["min", "max"]))

now_utc = pd.Timestamp.now(tz="UTC")
for col in date_cols:
    print(f"{col} 미래 날짜: {(df[col] > now_utc).sum()}건")

,created_at,opening_time
min,2023-04-28 12:27:23+00:00,2023-04-28 12:27:22+00:00
max,2024-05-07 11:32:30+00:00,2024-05-07 12:12:30+00:00


created_at 미래 날짜: 0건
opening_time 미래 날짜: 0건


## 범주형 컬럼 확인

In [7]:
df["status"].value_counts(dropna=False)

status
F    153411
O      4407
C       566
Name: count, dtype: int64

- `status`: C(닫힘), O(열림), F(종료)

## 질문 ID 리스트 확인

In [10]:
# 문자열(JSON)과 배열 형식을 모두 리스트로 통일
def parse_ids(value):
    try:
        result = json.loads(value) if isinstance(value, str) else list(value)
        return result if isinstance(result, list) else None
    except (TypeError, ValueError):
        return None

question_ids = df["question_piece_id_list"].apply(parse_ids)
valid_list = question_ids.notna()
item_count = question_ids.apply(lambda x: len(x) if isinstance(x, list) else pd.NA)
all_numeric = question_ids[valid_list].apply(
    lambda values: all(isinstance(item, Number) and not isinstance(item, bool) for item in values)
)

print("대괄호로 묶인 목록 형식이 아닌 행:", (~valid_list).sum())
print("빈 리스트:", item_count.eq(0).sum())
print("10개가 아닌 행:", (valid_list & item_count.ne(10)).sum())
print("숫자가 아닌 값이 포함된 행:", (~all_numeric).sum())
print("리스트 내부 중복:", question_ids[valid_list].apply(lambda x: len(x) != len(set(x))).sum())
print(item_count.describe())

대괄호로 묶인 목록 형식이 아닌 행: 0
빈 리스트: 0
10개가 아닌 행: 0
숫자가 아닌 값이 포함된 행: 0
리스트 내부 중복: 0
count    158384.0
mean         10.0
std           0.0
min          10.0
25%          10.0
50%          10.0
75%          10.0
max          10.0
Name: question_piece_id_list, dtype: float64


## 시간 순서 확인

In [9]:
# 질문 오픈 시간이 생성 시간보다 이른 행과 시간 차이 확인
early_open = df["opening_time"] < df["created_at"]
time_gap = (df.loc[early_open, "created_at"] - df.loc[early_open, "opening_time"]).dt.total_seconds()

print("오픈 시간이 이른 행:", early_open.sum())
print(time_gap.describe())

오픈 시간이 이른 행: 679
count    679.000000
mean       1.378498
std        1.923970
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       29.000000
dtype: float64


## 전처리 확인 결과

- 날짜 최솟값·최댓값과 미래 날짜 여부를 확인한다.
- `question_piece_id_list`는 유효한 배열이며 각 행에 질문 ID 10개가 저장되어 있는지 확인한다.
- 질문 ID가 모두 숫자형인지와 리스트 내부 중복 여부를 확인한다.
- `opening_time`이 `created_at`보다 최대 1~29초 이른 일부 행은 세션 처리 지연으로 판단하여 유지한다.
- 데이터 수정 여부는 확인 결과를 바탕으로 팀 협의 후 결정한다.